In [1]:
%cd ../..

/run/media/nazif/2F946E411BA61D49/mirscribe


In [2]:
import pandas as pd
from ast import literal_eval


In [3]:
def explode_lists(value):  # sourcery skip: inline-immediately-returned-variable
    if value == "not_found":
        return ["not_found"]
    
    try:
        value_list = literal_eval(value)
        
        return value_list
    
    except (SyntaxError, ValueError):
        return [value]

In [4]:
df2 = pd.read_csv("results/gain_pairs.csv")
df2 = (df2.assign(exploded=df2.ENST.apply(explode_lists))
       .explode("exploded")
       .drop(columns="ENST")
       .rename(columns={"exploded": "ENST"})
       )

df3 = pd.read_csv("results/loss_pairs.csv")
df3 = (df3.assign(exploded=df3.ENST.apply(explode_lists))
       .explode("exploded")
       .drop(columns="ENST")
       .rename(columns={"exploded": "ENST"})
       )


df2["gain"] = 1
df2["loss"] = 0

df3["gain"] = 0
df3["loss"] = 1

df = pd.concat([df2, df3], ignore_index=True)

df.head()

,id,mirna_accession,ENST
0,10_102293465_C_T,MIMAT0000062,ENST00000533589
1,10_102293465_C_T,MIMAT0000063,ENST00000533589
2,10_102293465_C_T,MIMAT0000064,ENST00000533589
3,10_102293465_C_T,MIMAT0000065,ENST00000533589
4,10_102293465_C_T,MIMAT0000259,ENST00000533589


In [11]:
# Analysis 4: Total Gain and Loss by MicroRNA
total_gain_loss_by_mirna = df.groupby('mirna_accession')[['gain', 'loss']].sum()


total_gain_loss_by_mirna["total"] = total_gain_loss_by_mirna["gain"] + total_gain_loss_by_mirna["loss"]

total_gain_loss_by_mirna.sort_values("total", ascending=False)

,gain,loss,total
mirna_accession,,,
MIMAT0021018,489,425,914
MIMAT0004543,463,395,858
MIMAT0026618,395,445,840
MIMAT0002858,381,435,816
MIMAT0004607,313,500,813
...,...,...,...
MIMAT0005937,9,0,9
MIMAT0019870,5,3,8
MIMAT0000459,4,1,5


In [14]:
# Assuming you already have your DataFrame 'df' with the required columns

# Group by 'id' and calculate the sum of 'gain' and 'loss' for each mutation
mutation_impact = df.groupby('id')[['gain', 'loss']].sum()

# Calculate the total impact (gain + loss) for each mutation
mutation_impact['total_impact'] = mutation_impact['gain'] + mutation_impact['loss']

# Sort the mutations by total impact in descending order
most_impactful_mutations = mutation_impact.sort_values(by='total_impact', ascending=False)

# Display the top most impactful mutations
print("Most Impactful Mutations:")
print(most_impactful_mutations.head())


Most Impactful Mutations:
                   gain  loss  total_impact
id                                         
5_138632501_A_G   11687  3844         15531
6_135515007_T_G    7258  4484         11742
17_56606430_G_T    6075  3550          9625
22_29099555_C_G    3818  3956          7774
12_102025497_G_A   1200  4760          5960


In [16]:
# Group by 'id' and calculate statistics for 'gain' and 'loss'
mutation_groups = df.groupby('id')[['gain', 'loss']].agg(['sum', 'mean', 'count'])

# Reset the column names for clarity
mutation_groups.columns = ['gain_total', 'gain_mean', 'gain_count', 'loss_total', 'loss_mean', 'loss_count']

# Calculate the total impact (gain + loss) for each mutation
mutation_groups['total_impact'] = mutation_groups['gain_total'] + mutation_groups['loss_total']

# Sort mutations by total impact in descending order
mutation_groups = mutation_groups.sort_values(by='total_impact', ascending=False)

# Print the top 10 most impactful mutations
top_10_impactful_mutations = mutation_groups.head(10)

# Print some cool insights
print("Top 10 Most Impactful Mutations:")
print(top_10_impactful_mutations)

# Calculate the number of mutations that only had gains or only had losses
mutations_with_only_gains = mutation_groups[mutation_groups['loss_total'] == 0]
mutations_with_only_losses = mutation_groups[mutation_groups['gain_total'] == 0]

# Print the number of mutations with only gains and only losses
print("\nNumber of Mutations with Only Gains:", len(mutations_with_only_gains))
print("Number of Mutations with Only Losses:", len(mutations_with_only_losses))

# Calculate the average gain and loss per mutation
average_gain_per_mutation = mutation_groups['gain_total'].mean()
average_loss_per_mutation = mutation_groups['loss_total'].mean()

# Print the average gain and loss per mutation
print("\nAverage Gain per Mutation:", average_gain_per_mutation)
print("Average Loss per Mutation:", average_loss_per_mutation)


Top 10 Most Impactful Mutations:
                                                    gain_total  gain_mean  \
id                                                                          
5_138632501_A_G                                          11687   0.752495   
6_135515007_T_G                                           7258   0.618123   
17_56606430_G_T                                           6075   0.631169   
22_29099555_C_G                                           3818   0.491124   
12_102025497_G_A                                          1200   0.201342   
5_67589535_A_G                                            4080   0.701031   
5_133476406_G_T                                           2772   0.486486   
17_7590693_ACCCAATCCAGGGAAGCGTGTCACCGTCGTGGAAAG...        1072   0.192287   
12_57926970_C_A                                           2660   0.544413   
21_35093482_G_A                                           1581   0.328622   

                                          

In [18]:

# Group mutations by 'mirna_accession' and calculate counts
mirna_grouped = df.groupby('mirna_accession').agg(
    total_mutations=pd.NamedAgg(column='id', aggfunc='count'),
    total_gains=pd.NamedAgg(column='gain', aggfunc='sum'),
    total_losses=pd.NamedAgg(column='loss', aggfunc='sum')
).reset_index().sort_values("total_mutations", ascending=False)

# Group mutations by 'ENST' (mRNA identifier) and calculate counts
mRNA_grouped = df.groupby('ENST').agg(
    total_mutations=pd.NamedAgg(column='id', aggfunc='count'),
    total_gains=pd.NamedAgg(column='gain', aggfunc='sum'),
    total_losses=pd.NamedAgg(column='loss', aggfunc='sum')
).reset_index().sort_values("total_mutations", ascending=False)

# Display the grouped data for microRNA and mRNA
print("MicroRNA Grouped Data:")
print(mirna_grouped.head())

print("\nmRNA Grouped Data:")
print(mRNA_grouped.head())


MicroRNA Grouped Data:
     mirna_accession  total_mutations  total_gains  total_losses
1711    MIMAT0021018              914          489           425
500     MIMAT0004543              858          463           395
2044    MIMAT0026618              840          395           445
288     MIMAT0002858              816          381           435
548     MIMAT0004607              813          313           500

mRNA Grouped Data:
                 ENST  total_mutations  total_gains  total_losses
2896        not_found            62192        30537         31655
151   ENST00000269305            11086         4311          6775
1293  ENST00000445888            10079         4173          5906
1051  ENST00000420246            10070         4173          5897
1363  ENST00000455263            10070         4173          5897


In [49]:
# Assuming you have loaded the dataset into a DataFrame named "df"

# Group mutations by 'mirna_accession' and calculate the total gains, losses, and total mutations
mirna_trends = df.groupby('mirna_accession').agg(
    total_gains=pd.NamedAgg(column='gain', aggfunc='sum'),
    total_losses=pd.NamedAgg(column='loss', aggfunc='sum'),
    total_modifications=pd.NamedAgg(column='id', aggfunc='count')
).reset_index()


# Group mutations by 'ENST' (mRNA identifier) and calculate the total gains, losses, and total mutations
mRNA_trends = df.groupby('ENST').agg(
    total_gains=pd.NamedAgg(column='gain', aggfunc='sum'),
    total_losses=pd.NamedAgg(column='loss', aggfunc='sum'),
    total_modifications=pd.NamedAgg(column='id', aggfunc='count')
).reset_index()




In [50]:
mRNA_trends.sort_values("total_modifications", ascending=False)

,ENST,total_gains,total_losses,total_modifications
2896,not_found,30537,31655,62192
151,ENST00000269305,4311,6775,11086
1293,ENST00000445888,4173,5906,10079
1051,ENST00000420246,4173,5897,10070
1363,ENST00000455263,4173,5897,10070
...,...,...,...,...
350,ENST00000334602,2,1,3
149,ENST00000268957,2,1,3
639,ENST00000375459,1,2,3
1701,ENST00000490709,0,1,1


In [55]:
from scripts.ensembl import *

g37 = import_pyensembl(37)
ensts = mRNA_trends[mRNA_trends.ENST != "not_found"].ENST.to_list()

gene_dict = {}

for i in ensts:
    gene_name = g37.gene_name_of_transcript_id(i)
    gene_dict[i] = gene_name
    
mRNA_trends['gene_name'] = mRNA_trends['ENST'].map(gene_dict)

In [82]:
mRNA_trends[mRNA_trends.ENST != "not_found"].sort_values("total_modifications", ascending=False).head(20)


,ENST,total_gains,total_losses,total_modifications,gene_name,in_gene_symbols
151,ENST00000269305,4311,6775,11086,TP53,0
1293,ENST00000445888,4173,5906,10079,TP53,0
1363,ENST00000455263,4173,5897,10070,TP53,0
1051,ENST00000420246,4173,5897,10070,TP53,0
1856,ENST00000509690,3317,4842,8159,TP53,0
998,ENST00000413465,3497,4340,7837,TP53,0
494,ENST00000359597,3497,4340,7837,TP53,0
1867,ENST00000510385,3056,3848,6904,TP53,0
1806,ENST00000504290,3056,3848,6904,TP53,0
1814,ENST00000504937,3056,3848,6904,TP53,0


# GSEA json

In [85]:
import json

# Specify the path to your JSON file
json_file_path = "data/gsea/HALLMARK_DNA_REPAIR.v2023.1.Hs.json"

# Open and read the JSON file
with open(json_file_path, "r") as json_file:
    data = json.load(json_file)

# Extract gene symbols
if "HALLMARK_DNA_REPAIR" in data:
    gene_symbols = data["HALLMARK_DNA_REPAIR"].get("geneSymbols", [])
else:
    gene_symbols = []

# gene_symbols now contains the list of gene symbols
print(gene_symbols)


['AAAS', 'ADA', 'ADCY6', 'ADRM1', 'AGO4', 'AK1', 'AK3', 'ALYREF', 'APRT', 'ARL6IP1', 'BCAM', 'BCAP31', 'BOLA2', 'BRF2', 'CANT1', 'CCNO', 'CDA', 'CETN2', 'CLP1', 'CMPK2', 'COX17', 'CSTF3', 'DAD1', 'DCTN4', 'DDB1', 'DDB2', 'DGCR8', 'DGUOK', 'DUT', 'EDF1', 'EIF1B', 'ELL', 'ELOA', 'ERCC1', 'ERCC2', 'ERCC3', 'ERCC4', 'ERCC5', 'ERCC8', 'FEN1', 'GMPR2', 'GPX4', 'GSDME', 'GTF2A2', 'GTF2B', 'GTF2F1', 'GTF2H1', 'GTF2H3', 'GTF2H5', 'GTF3C5', 'GUK1', 'HCLS1', 'HPRT1', 'IMPDH2', 'ITPA', 'LIG1', 'MPC2', 'MPG', 'MRPL40', 'NCBP2', 'NELFB', 'NELFCD', 'NELFE', 'NFX1', 'NME1', 'NME3', 'NME4', 'NPR2', 'NT5C', 'NT5C3A', 'NUDT21', 'NUDT9', 'PCNA', 'PDE4B', 'PDE6G', 'PNP', 'POLA1', 'POLA2', 'POLB', 'POLD1', 'POLD3', 'POLD4', 'POLE4', 'POLH', 'POLL', 'POLR1C', 'POLR1D', 'POLR1H', 'POLR2A', 'POLR2C', 'POLR2D', 'POLR2E', 'POLR2F', 'POLR2G', 'POLR2H', 'POLR2I', 'POLR2J', 'POLR2K', 'POLR3C', 'POLR3GL', 'POM121', 'PRIM1', 'RAD51', 'RAD52', 'RAE1', 'RALA', 'RBX1', 'REV3L', 'RFC2', 'RFC3', 'RFC4', 'RFC5', 'RNMT', 'R

In [87]:

# Create a new column based on the condition
mRNA_trends['in_gene_symbols'] = mRNA_trends['gene_name'].apply(lambda x: 1 if x in gene_symbols else 0)

mRNA_trends[mRNA_trends.in_gene_symbols == 1].sort_values("total_modifications", ascending=False).head(20)


,ENST,total_gains,total_losses,total_modifications,gene_name,in_gene_symbols
151,ENST00000269305,4311,6775,11086,TP53,1
1293,ENST00000445888,4173,5906,10079,TP53,1
1051,ENST00000420246,4173,5897,10070,TP53,1
1363,ENST00000455263,4173,5897,10070,TP53,1
1856,ENST00000509690,3317,4842,8159,TP53,1
998,ENST00000413465,3497,4340,7837,TP53,1
494,ENST00000359597,3497,4340,7837,TP53,1
1814,ENST00000504937,3056,3848,6904,TP53,1
1806,ENST00000504290,3056,3848,6904,TP53,1
1867,ENST00000510385,3056,3848,6904,TP53,1


In [73]:
hallmark["geneSymbols"]

KeyError: 'geneSymbols'

In [ ]:
grouped = df.groupby("ENST")

In [ ]:
unique_microRNAs_per_transcript = grouped['mirna_accession'].nunique()
unique_microRNAs_per_transcript.sort_values()

In [ ]:
df[df.ENST == "ENST00000269305"].id.nunique()

In [ ]:
mutations_with_most_microRNA_binding = df.groupby('id')['mirna_accession'].count().sort_values(ascending=False)

mutations_with_most_microRNA_binding

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = df.pivot_table(index='id', columns='mirna_accession', aggfunc='size', fill_value=0)
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, cmap='viridis', cbar=True)
plt.xlabel('MicroRNA Accession')
plt.ylabel('Mutation ID')
plt.title('Mutation-MicroRNA Binding Heatmap')
plt.show()


In [ ]:
# Group the DataFrame by mutations (unique 'id' values)
grouped_by_mutations = df.groupby('id')

# Calculate the number of unique miRNA accessions per mutation
unique_miRNA_counts = grouped_by_mutations['mirna_accession'].nunique()

# Calculate the number of unique mRNA (ENST) per mutation
unique_mRNA_counts = grouped_by_mutations['ENST'].nunique()

# Calculate the total number of interactions per mutation
total_interactions = grouped_by_mutations.size()

# Calculate the mutations that introduced new interactions
mutations_introducing_new_interactions = total_interactions[total_interactions > 1]

# Calculate the mutations with the most unique miRNA accessions
mutations_with_most_unique_miRNAs = unique_miRNA_counts.idxmax()
most_unique_miRNA_count = unique_miRNA_counts.max()

# Calculate the mutations with the most unique mRNA (ENST)
mutations_with_most_unique_mRNAs = unique_mRNA_counts.idxmax()
most_unique_mRNA_count = unique_mRNA_counts.max()

# Display some cool insights
print("Cool Insights from Mutations Data:")
print(f"Total number of unique mutations: {len(grouped_by_mutations)}")
print(f"Mutations introducing new interactions: {len(mutations_introducing_new_interactions)}")
print(f"Mutation with the most unique miRNA accessions ({most_unique_miRNA_count}): {mutations_with_most_unique_miRNAs}")
print(f"Mutation with the most unique mRNA (ENST) ({most_unique_mRNA_count}): {mutations_with_most_unique_mRNAs}")


In [ ]:
# Group the DataFrame by mutation ('id') and mRNA (ENST)
mutation_mRNA_counts = df.groupby(['id', 'ENST']).size().reset_index(name='count')

# Sort the pairs by the count of occurrences in descending order
sorted_pairs = mutation_mRNA_counts.sort_values(by='count', ascending=False)

# Display the mutation-mRNA pairs that have been affected the most
most_affected_pairs = sorted_pairs  # You can adjust the number of pairs to display
print("Most affected mutation-mRNA pairs:")
print(most_affected_pairs)


In [ ]:
# Group the most affected pairs by the same 'id' and the same count
grouped_most_affected = most_affected_pairs.groupby(['id', 'count'])

# Display the grouped most affected mutation-mRNA pairs with ENST values
for (id_value, count), group_data in grouped_most_affected:
    print(f"Mutation 'id': {id_value}, Count: {count}")
    print("Affected mRNA (ENST) values:")
    print(group_data['ENST'].values)
    print("\n")


In [ ]:
# Sort the pairs by the count of occurrences in ascending order
least_affected_pairs = mutation_mRNA_counts.sort_values(by='count', ascending=True)

# Display the mutation-mRNA pairs that have been affected the least
least_affected_pairs = least_affected_pairs.head(10)  # You can adjust the number of pairs to display
print("Least affected mutation-mRNA pairs:")
print(least_affected_pairs)


In [ ]:


# Group the DataFrame by mutation ('id') and count the occurrences of each pair
mutation_mRNA_counts = df.groupby(['id', 'ENST']).size().reset_index(name='count')

# Sort the pairs by the count of occurrences in descending order
sorted_pairs = mutation_mRNA_counts.sort_values(by='count', ascending=False)

# Group the most affected pairs by the same 'id' and the same count
grouped_most_affected = sorted_pairs.groupby(['id', 'count'])['ENST'].apply(list).reset_index()

# Display the DataFrame
print("DataFrame of most affected mutation-mRNA pairs with ENST values in lists:")

grouped_most_affected.sort_values(by="count",)